In [1]:
import pandas as pd  # For data manipulation and DataFrames
import re  # For Regular Expressions (crucial for text cleaning)
from datasets import load_dataset  # The easiest way to pull the FAQ data

import nltk
import spacy
from nltk.corpus import stopwords

# NLP & Preprocessing
# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nlp = spacy.load("en_core_web_sm")

N:\Study\Material\DEPI-R4\lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Moham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Moham\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
dataset = load_dataset("MakTek/Customer_support_faqs_dataset")
df = dataset['train'].to_pandas()

Generating train split: 100%|██████████| 200/200 [00:00<00:00, 12785.37 examples/s]


In [3]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  200 non-null    object
 1   answer    200 non-null    object
dtypes: object(2)
memory usage: 3.3+ KB


,question,answer
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...


### Data Cleaning

In [4]:
df.duplicated().sum()

np.int64(111)

In [5]:
df.isnull().sum()

question    0
answer      0
dtype: int64

In [6]:
df.drop_duplicates().copy()

,question,answer
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...
...,...,...
84,What are your business hours?,Our business hours are [working hours]. During...
85,Do you offer a satisfaction guarantee?,"Yes, we offer a satisfaction guarantee on our ..."
86,How can I apply for a job at your company?,"To apply for a job at our company, visit our C..."
87,What is the warranty on your products?,The warranty on our products varies by item. P...


In [7]:
def clean_text(text):
    if not isinstance(text, str):
        return str(text)

    text = text.strip()  # Strip extra whitespace

    # Remove excessive spaces between words
    text = re.sub(r'\s+', ' ', text)

    # Standardize punctuation (e.g., multiple periods or question marks)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r'\?{2,}', '?', text)
    text = re.sub(r'!{2,}', '!', text)

    # Ensure Capitalization starts
    if len(text) > 0:
        text = text[0].upper() + text[1:]

    return text


# 2. Apply the function to your DataFrame's columns
df['question'] = df['question'].apply(clean_text)
df['answer'] = df['answer'].apply(clean_text)

# Drop any duplicate questions
df['q_lower'] = df['question'].str.lower()
df = df.drop_duplicates(subset=['q_lower']).drop(columns=['q_lower'])

df.head()

,question,answer
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...


### Structure & Organization (Categorization)

In [8]:
def categorize(qa):
    q, a = qa[0].lower(), qa[1].lower()
    text = q + " " + a
    if any(word in text for word in ['account', 'password', 'login', 'sign up']):
        return 'Account & Profile'
    elif any(word in text for word in ['payment', 'credit card', 'debit', 'pay', 'billing']):
        return 'Billing & Payments'
    elif any(word in text for word in ['order', 'track', 'shipping', 'delivery', 'cancel']):
        return 'Orders & Shipping'
    elif any(word in text for word in ['return', 'refund', 'policy', 'exchange']):
        return 'Returns & Refunds'
    elif any(word in text for word in ['contact', 'support', 'help', 'email']):
        return 'Customer Support'
    else:
        return 'General/Product Inquiry'


df['category'] = df[['question', 'answer']].apply(categorize, axis=1)


def assign_difficulty(answer):
    words = len(answer.split())
    if words < 20:
        return 'Basic'
    elif words < 30:
        return 'Intermediate'
    else:
        return 'Advanced'


df['difficulty_level'] = df['answer'].apply(assign_difficulty)
df['metadata_tags'] = ''
df.head()

C:\Users\Moham\AppData\Local\Temp\ipykernel_23160\3319116985.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  q, a = qa[0].lower(), qa[1].lower()


,question,answer,category,difficulty_level,metadata_tags
0,How can I create an account?,"To create an account, click on the 'Sign Up' b...",Account & Profile,Intermediate,
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and...",Billing & Payments,Basic,
2,How can I track my order?,You can track your order by logging into your ...,Account & Profile,Intermediate,
3,What is your return policy?,Our return policy allows you to return product...,Returns & Refunds,Advanced,
4,Can I cancel my order?,You can cancel your order if it has not been s...,Orders & Shipping,Advanced,


In [10]:
def build_metadata_tags_faq(row):
    tags = [
        f"category:{row['category']}",
        f"difficulty:{row['difficulty_level']}",
        "source:faq"
    ]
    return " | ".join(tags)

df["metadata_tags"] = df.apply(build_metadata_tags_faq, axis=1)

### Text Standardization (Abbreviations)

In [11]:
# Expand common abbreviations and normalize terms
abbreviations = {
    r'\bfaq\'s?\b': 'Frequently Asked Questions',
    r'\binfo\b': 'information',
    r'\bpls\b': 'please',
    r'\basap\b': 'as soon as possible',
    r'\bcs\b': 'customer service',
}
for abbr, exp in abbreviations.items():
    df['question'] = df['question'].str.replace(abbr, exp, flags=re.IGNORECASE, regex=True)
    df['answer'] = df['answer'].str.replace(abbr, exp, flags=re.IGNORECASE, regex=True)
df.head()

,question,answer,category,difficulty_level,metadata_tags
0,How can I create an account?,"To create an account, click on the 'Sign Up' b...",Account & Profile,Intermediate,category:Account & Profile | difficulty:Interm...
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and...",Billing & Payments,Basic,category:Billing & Payments | difficulty:Basic...
2,How can I track my order?,You can track your order by logging into your ...,Account & Profile,Intermediate,category:Account & Profile | difficulty:Interm...
3,What is your return policy?,Our return policy allows you to return product...,Returns & Refunds,Advanced,category:Returns & Refunds | difficulty:Advanc...
4,Can I cancel my order?,You can cancel your order if it has not been s...,Orders & Shipping,Advanced,category:Orders & Shipping | difficulty:Advanc...


### Quality Checks & Export

In [12]:
df['requires_review'] = False
df['review_reason'] = ''


def quality_check(row):
    reasons = []
    if len(row['question'].split()) < 3:
        reasons.append("Question too short")
    if len(row['answer'].split()) < 5:
        reasons.append("Answer lacks detail")
    if not row['question'].endswith('?'):
        reasons.append("Question lacks question mark")

    if reasons:
        return pd.Series([True, "; ".join(reasons)])
    return pd.Series([False, ""])


df[['requires_review', 'review_reason']] = df.apply(quality_check, axis=1)

# Reorder columns and save to CSV!
df = df[['category', 'difficulty_level', 'question', 'answer', 'requires_review', 'review_reason', 'metadata_tags']]
df.to_csv('../data/cleaned_faq_dataset.csv', index=False)
df.head()

,category,difficulty_level,question,answer,requires_review,review_reason,metadata_tags
0,Account & Profile,Intermediate,How can I create an account?,"To create an account, click on the 'Sign Up' b...",False,,category:Account & Profile | difficulty:Interm...
1,Billing & Payments,Basic,What payment methods do you accept?,"We accept major credit cards, debit cards, and...",False,,category:Billing & Payments | difficulty:Basic...
2,Account & Profile,Intermediate,How can I track my order?,You can track your order by logging into your ...,False,,category:Account & Profile | difficulty:Interm...
3,Returns & Refunds,Advanced,What is your return policy?,Our return policy allows you to return product...,False,,category:Returns & Refunds | difficulty:Advanc...
4,Orders & Shipping,Advanced,Can I cancel my order?,You can cancel your order if it has not been s...,False,,category:Orders & Shipping | difficulty:Advanc...
